# 05 — Eval harness

Run one model against one task over the full item set.

**Order matters:** write the manifest before the run starts, and write each raw response before parsing it. Scoring bugs are recoverable; lost responses mean re-paying for the run.

No sampling. Full pinned model strings only. Read the `running-evals` skill.

In [ ]:
# Colab bootstrap. Run once per runtime.
!pip install -q google-cloud-storage

from google.colab import auth
auth.authenticate_user()

import sys, pathlib
GITHUB_USER = ''   # TODO
REPO = pathlib.Path('/content/auditagent-bench')
if not REPO.exists():
    !git clone -q https://github.com/{GITHUB_USER}/auditagent-bench.git {REPO}
sys.path.insert(0, str(REPO))

In [ ]:
# --- Config. Every tunable value in this notebook lives in this cell. ---
BUCKET = ''            # TODO: GCS bucket name
BENCHMARK_PREFIX = 'benchmark'
RESULTS_PREFIX = 'results'
DATASET_VERSION = 'v0.1'

TASK = 'C'
MODEL_STRING = ''      # TODO: full pinned identifier, never an alias or 'latest'
VENDOR = ''            # anthropic | google | openai | vertex-model-garden
PROMPT_VERSION = 'v1'
PROMPT_FILE = 'prompts/v1/task_c_clean_dirty.md'
USE_BATCH = True

# API keys come from Colab Secrets, never a cell literal.
# from google.colab import userdata; userdata.get('ANTHROPIC_API_KEY')

In [ ]:
import hashlib, json, pathlib
from datetime import datetime, timezone
from src import gcs

assert MODEL_STRING and 'latest' not in MODEL_STRING, \
    'MODEL_STRING must be a full pinned identifier'

template = (REPO / PROMPT_FILE).read_text(encoding='utf-8')
prompt_sha = hashlib.sha256(template.encode()).hexdigest()

items = [i for i in gcs.read_jsonl(BUCKET, f'{BENCHMARK_PREFIX}/{DATASET_VERSION}/items.jsonl')
         if i['task'] == TASK]

run_id = f"{datetime.now(timezone.utc):%Y-%m-%dT%H-%M}_task{TASK}_{MODEL_STRING}"
manifest = {
    'run_id': run_id, 'task': TASK, 'model_string': MODEL_STRING, 'vendor': VENDOR,
    'prompt_version': PROMPT_VERSION, 'prompt_sha256': prompt_sha,
    'dataset_version': DATASET_VERSION, 'n_items': len(items),
    'run_started_at': datetime.now(timezone.utc).isoformat(), 'batch': USE_BATCH,
}

# Written BEFORE the run. A result without a manifest is unpublishable.
gcs.write_json(BUCKET, f'{RESULTS_PREFIX}/{run_id}/manifest.json', manifest)
print(json.dumps(manifest, indent=2))

In [ ]:
# TODO: implement the vendor call. Prefer the batch endpoint — nothing here is
# latency-sensitive and batch pricing roughly halves cost.
#
# Contract for whatever goes here:
#   1. write the raw response before parsing it
#   2. resume by item_id — batch jobs fail partway
#   3. full item set, no sampling

def build_prompt(item):
    body = template.split('---', 2)[-1]
    return body.replace('{{disclosure_text}}', item['input']['disclosure_text'])


def call_model(prompt: str) -> str:
    raise NotImplementedError(f'wire up the vendor SDK for {VENDOR}')

In [ ]:
from src.parsing import parse_json_response

REQUIRED = {'A': {'severity', 'coso_component'},
            'B': {'designed_effectively'},
            'C': {'is_deficient'},
            'D': {'citation'}}[TASK]

scored, parse_failed = [], 0

for item in items:
    raw_path = f'{RESULTS_PREFIX}/{run_id}/raw/{item["item_id"]}.json'
    if gcs.blob_exists(BUCKET, raw_path):
        raw = gcs.read_json(BUCKET, raw_path)['text']   # resume
    else:
        raw = call_model(build_prompt(item))
        gcs.write_json(BUCKET, raw_path, {'text': raw})  # persist before parsing

    result = parse_json_response(raw, REQUIRED)
    if not result.ok:
        parse_failed += 1
    scored.append({'item_id': item['item_id'], 'label': item['label'],
                   'parsed': result.value, 'parse_error': result.error})

gcs.write_jsonl(BUCKET, f'{RESULTS_PREFIX}/{run_id}/scored.jsonl', scored)
print(f'items={len(scored)} parse_failed={parse_failed} ({parse_failed / len(scored):.1%})')
print('parse_failed is a reported metric, not an error to hide.')